In [15]:
import os
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [16]:
PROJECT_DIR = Path.cwd()

DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"
PLOTS_DIR = PROJECT_DIR / "plots"
REPORTS_DIR = PROJECT_DIR / "reports"
LOGS_DIR = PROJECT_DIR / "logs"

CHUNK_SIZE = 512
CHUNK_OVERLAP = 100

TOP_K = 5

print(f"Project Directory : {PROJECT_DIR}")
print(f"Chunk Size        : {CHUNK_SIZE}")
print(f"Chunk Overlap     : {CHUNK_OVERLAP}")
print(f"Top K             : {TOP_K}")

Project Directory : c:\Users\premchand\Desktop\Rag_chatbot\experiments
Chunk Size        : 512
Chunk Overlap     : 100
Top K             : 5


In [17]:
def start_timer():
    return perf_counter()


def stop_timer(start_time):
    return round(perf_counter() - start_time, 4)


def print_section(title):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)

In [18]:
print_section("Loading PDF")

pdf_path = DATA_DIR / "sample.pdf"

start = start_timer()

loader = PyPDFLoader(str(pdf_path))
documents = loader.load()

loading_time = stop_timer(start)

print(f"PDF Path      : {pdf_path}")
print(f"Total Pages   : {len(documents)}")
print(f"Loading Time  : {loading_time} sec")


Loading PDF
PDF Path      : c:\Users\premchand\Desktop\Rag_chatbot\experiments\data\sample.pdf
Total Pages   : 177
Loading Time  : 1.7594 sec


In [21]:
print_section("Chunking Documents")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

start = start_timer()

document_chunks = text_splitter.split_documents(documents)

chunking_time = stop_timer(start)

print(f"Total Chunks  : {len(document_chunks)}")
print(f"Chunking Time : {chunking_time} sec")

print("\nFirst Chunk Metadata")
print(document_chunks[0].metadata)

print("\nFirst Chunk Length")
print(len(document_chunks[0].page_content))

print("\nFirst Chunk Preview")
print("-" * 60)
print(document_chunks[0].page_content[:300])


Chunking Documents
Total Chunks  : 559
Chunking Time : 0.0096 sec

First Chunk Metadata
{'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2023-01-04T05:56:26+00:00', 'source': 'c:\\Users\\premchand\\Desktop\\Rag_chatbot\\experiments\\data\\sample.pdf', 'total_pages': 177, 'page': 0, 'page_label': '1'}

First Chunk Length
483

First Chunk Preview
------------------------------------------------------------
ARTIFICIAL INTELLIGENCE 
& 
 MACHINE LEARNING 
          Lecture Notes 
B.TECH 
(III YEAR – II SEM) 
(2022-2023) 
Prepared by:  
                        Ms.Anitha Patibandla, Associate Professor 
                  Dr.B.Jyothi, Professor 
                                                    Ms.K.Bhava


In [22]:
# ==========================================
# Load Embedding Model
# ==========================================

from sentence_transformers import SentenceTransformer

print_section("Loading Embedding Model")

MODEL_NAME = "BAAI/bge-base-en-v1.5"

start = start_timer()

embedding_model = SentenceTransformer(MODEL_NAME)

loading_time = stop_timer(start)

print(f"Model Name   : {MODEL_NAME}")
print(f"Loading Time : {loading_time} sec")

c:\Users\premchand\Desktop\Rag_chatbot\experiments\exp_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loading Embedding Model


c:\Users\premchand\Desktop\Rag_chatbot\experiments\exp_venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\premchand\.cache\huggingface\hub\models--BAAI--bge-base-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19499.73it/s]


Model Name   : BAAI/bge-base-en-v1.5
Loading Time : 26.9443 sec


In [27]:
# ==========================================
# Generate Embeddings
# ==========================================

print_section("Generating Embeddings")

chunk_texts = [doc.page_content for doc in document_chunks]
chunk_metadata = [doc.metadata for doc in document_chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)
embedding_time = stop_timer(start)

print(f"Total Chunks        : {len(document_chunks)}")
print(f"Embedding Shape     : {embeddings.shape}")
print(f"Embedding Dimension : {embeddings.shape[1]}")
print(f"Embedding Time      : {embedding_time} sec")

print("\nFirst Embedding (First 10 Values)")
print(embeddings[0][:10])


Generating Embeddings


Batches: 100%|██████████| 18/18 [00:40<00:00,  2.27s/it]

Total Chunks        : 559
Embedding Shape     : (559, 768)
Embedding Dimension : 768
Embedding Time      : 108.9222 sec

First Embedding (First 10 Values)
[ 0.021401   -0.00333653 -0.00868127 -0.00279708  0.06761926  0.03447151
  0.03383594 -0.00545449 -0.01088902 -0.00483813]


In [28]:
# ==========================================
# FAISS - Flat Index
# ==========================================

import faiss

print_section("Building FAISS Flat Index")

embedding_dimension = embeddings.shape[1]

start = start_timer()

faiss_flat_index = faiss.IndexFlatL2(embedding_dimension)

faiss_flat_index.add(embeddings)

index_build_time = stop_timer(start)

print(f"Index Type        : {type(faiss_flat_index).__name__}")
print(f"Embedding Dimension : {embedding_dimension}")
print(f"Vectors Stored      : {faiss_flat_index.ntotal}")
print(f"Build Time          : {index_build_time} sec")


Building FAISS Flat Index
Index Type        : IndexFlatL2
Embedding Dimension : 768
Vectors Stored      : 559
Build Time          : 0.0008 sec
